# Othello DQN on Google Colab — Universal Game Engine

Universal Game Engine のバックエンド（Bun + gRPC）を **Colab 内で起動**し、
Python (PyTorch) の DQN エージェントが gRPC の `Reset` / `Step` で自己対戦しながら学習します。
学習済みモデルは Google Drive に保存されます。

**手順**: ランタイム → 「ランタイムのタイプを変更」で GPU (T4) を選んでから、上から順に実行してください。

| ステップ | 内容 |
| --- | --- |
| 1 | Google Drive をマウント（モデル保存先） |
| 2 | リポジトリを clone、Bun をインストール |
| 3 | バックエンドを `RL_MODE=true` でバックグラウンド起動 |
| 4 | 学習 (`uge_rl.train`) |
| 5 | 評価 (`uge_rl.evaluate`) と学習曲線 |


## 1. Google Drive をマウント

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MODEL_DIR = '/content/drive/MyDrive/UniversalGameEngine/models'
import os; os.makedirs(MODEL_DIR, exist_ok=True)
print('models will be saved to', MODEL_DIR)

## 2. リポジトリの取得と Bun のインストール

private リポジトリの場合は `REPO_URL` を `https://<GITHUB_TOKEN>@github.com/...` の形式にしてください。

In [ ]:
import os
REPO_URL = 'https://github.com/takumi-mr/UniversalGameEngine.git'  #@param {type:"string"}
BRANCH = 'main'  #@param {type:"string"}

if not os.path.exists('/content/UniversalGameEngine'):
    !git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/UniversalGameEngine
%cd /content/UniversalGameEngine

# Bun
!curl -fsSL https://bun.sh/install | bash > /dev/null 2>&1
os.environ['PATH'] = '/root/.bun/bin:' + os.environ['PATH']
!bun --version

# 依存関係（postinstall の git hook 設定は Colab では不要なのでスキップ）
!bun install --frozen-lockfile --ignore-scripts

## 3. バックエンドをバックグラウンド起動

`RL_MODE=true` にすると Redis / MongoDB なしのインメモリ動作になります。ログは `server.log` に出ます。

In [ ]:
import subprocess, sys, time
sys.path.insert(0, '/content/UniversalGameEngine/apps/ml')

env = dict(os.environ, RL_MODE='true', PORT='3000', GRPC_PORT='50051')
server = subprocess.Popen(
    ['bun', 'run', 'apps/backend/server.ts'],
    cwd='/content/UniversalGameEngine',
    env=env,
    stdout=open('/content/server.log', 'w'),
    stderr=subprocess.STDOUT,
)

from uge_rl.env import wait_for_server
wait_for_server('localhost:50051', timeout_sec=90)
print('gRPC server ready (pid', server.pid, ')')
!tail -n 5 /content/server.log

## 4. Python 依存関係

In [ ]:
!pip install -q -r apps/ml/requirements.txt
import torch; print('torch', torch.__version__, 'cuda:', torch.cuda.is_available())

## 5. 学習

`--episodes` を増やすほど強くなります（T4 で 1 エピソード ≒ 0.3〜0.5 秒）。
途中経過のチェックポイントも `--save-every` ごとに同じパスへ上書き保存されます。
中断した場合は `--resume {MODEL_PATH}` を付けて再開できます。

In [ ]:
EPISODES = 3000  #@param {type:"integer"}
MODEL_PATH = f'{MODEL_DIR}/othello_dqn.pt'

!cd apps/ml && python -m uge_rl.train     --game othello     --address localhost:50051     --episodes {EPISODES}     --out {MODEL_PATH}     --eval-every 200 --eval-games 20     --save-every 500 --log-every 50

## 6. 評価（ランダムプレイヤーとの対戦）

In [ ]:
!cd apps/ml && python -m uge_rl.evaluate --checkpoint {MODEL_PATH} --address localhost:50051 --games 200

## 7. 学習曲線（対ランダム勝率）

In [ ]:
import json
import matplotlib.pyplot as plt

meta = json.load(open(MODEL_PATH.replace('.pt', '.json')))
hist = meta.get('eval_history', [])
if hist:
    ep, wr = zip(*hist)
    plt.plot(ep, wr, marker='o')
    plt.axhline(0.5, ls='--', c='gray')
    plt.xlabel('episode'); plt.ylabel('win rate vs random'); plt.ylim(0, 1)
    plt.title(f"Othello DQN ({meta['total_steps']} steps)")
    plt.show()
print({k: meta[k] for k in ('game_type', 'arch', 'total_steps', 'train_steps', 'saved_at', 'git_commit')})

## 8. 保存されたファイル

- `othello_dqn.pt` — PyTorch の state_dict + メタ情報（`uge_rl.checkpoint.load_checkpoint` で復元）
- `othello_dqn.json` — メタ情報のみ（ゲーム種別・観測形状・行動数・学習ステップ数・学習曲線）

In [ ]:
!ls -la {MODEL_DIR}

## 9. 後片付け（任意）

In [ ]:
server.terminate()
print('server stopped')